# ASR RTF Benchmark — 4 Vietnamese Models

Đo **RTF (Real-Time Factor)** trên A100 cho 4 model từ bảng WER paper.
WER lấy từ paper — notebook này chỉ đo RTF.

| Model | Params | Train data | WER VLSP2020-T1 (paper) |
|-------|--------|------------|-------------------------|
| ZipFormer-30M | 30M | 6,000h | 12.29% |
| ChunkFormer-110M | 110M | 3,000h | 14.09% |
| PhoWhisper-Large | ~1.5B | 800h | 13.75% |
| VietASR-ZipFormer-68M | 68M | 70,000h | 14.45% |

**Dataset:** 400 samples từ `doof-ferb/vlsp2020_vinai_100h` (dur 2–20s, seed=42)  
**Metrics:** RTF P50, RTF P95  
**Chạy trên:** Google Colab A100 40GB

**Chuẩn bị trước:**
1. Chạy `scripts/download_asr_benchmark_data.py` trên máy local
2. Upload lên Drive:
   - `Drive/asr_benchmark/samples/`  ← thư mục WAV (400 files)
   - `Drive/asr_benchmark/asr_benchmark_samples.jsonl`  ← metadata

## 0. Install

In [ ]:
%%capture
!pip install sherpa-onnx soundfile numpy
!pip install chunkformer
!pip install transformers accelerate torch torchaudio
!pip install huggingface_hub

## 1. Mount Drive & load samples

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import numpy as np
import soundfile as sf
import tempfile, os
from pathlib import Path

DRIVE_DIR   = '/content/drive/MyDrive/asr_benchmark'
SAMPLES_DIR = Path(f'{DRIVE_DIR}/samples')
JSONL_PATH  = f'{DRIVE_DIR}/asr_benchmark_samples.jsonl'

# Load metadata
with open(JSONL_PATH, encoding='utf-8') as f:
    meta = [json.loads(l) for l in f if l.strip()]

# Load audio arrays từ WAV files
print(f'Loading {len(meta)} WAV files from Drive...')
samples = []
for i, m in enumerate(meta):
    wav_path = SAMPLES_DIR / m['wav_file']
    arr, sr = sf.read(str(wav_path), dtype='float32')
    samples.append({
        'idx':      m['idx'],
        'array':    arr,
        'sr':       sr,
        'duration': m['duration'],
        'sentence': m['sentence'],
    })
    if (i + 1) % 100 == 0:
        print(f'  loaded {i+1}/{len(meta)}')

durations = [s['duration'] for s in samples]
print(f'\nLoaded {len(samples)} samples')
print(f'Duration — min={min(durations):.1f}s  mean={np.mean(durations):.1f}s  '
      f'median={float(np.median(durations)):.1f}s  max={max(durations):.1f}s')
print(f'Total: {sum(durations)/60:.1f} min audio')

# Tmpdir cho ChunkFormer (cần path file wav)
_tmpdir = tempfile.mkdtemp()
_tmp_wav = os.path.join(_tmpdir, 'tmp.wav')

## 2. RTF measurement helper

In [ ]:
import time

def measure_rtf(infer_fn, samples, model_name, warmup=5):
    """
    infer_fn(array: np.ndarray, sr: int) → None
    samples: list of dict với keys 'array', 'sr', 'duration'
    """
    print(f'\n[{model_name}] Warmup ({warmup} samples)...')
    for s in samples[:warmup]:
        infer_fn(s['array'], s['sr'])

    print(f'[{model_name}] Measuring RTF ({len(samples)} samples)...')
    rtf_values = []
    for i, s in enumerate(samples):
        t0 = time.perf_counter()
        infer_fn(s['array'], s['sr'])
        elapsed = time.perf_counter() - t0
        rtf_values.append(elapsed / s['duration'])

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(samples)}  running P50={np.percentile(rtf_values, 50):.4f}')

    p50 = float(np.percentile(rtf_values, 50))
    p95 = float(np.percentile(rtf_values, 95))
    print(f'[{model_name}] → P50={p50:.4f}  P95={p95:.4f}')
    return {'model': model_name, 'rtf_values': rtf_values, 'p50': p50, 'p95': p95}

all_results = {}

## 3. Model 1 — ZipFormer-30M (sherpa-onnx, CUDA)

In [ ]:
import sherpa_onnx
from huggingface_hub import snapshot_download

print("Downloading ZipFormer-30M ONNX files...")
zipformer_30m_dir = snapshot_download("hynt/Zipformer-30M-RNNT-6000h")
print(f"Downloaded to: {zipformer_30m_dir}")

import os
# List files to find correct names
for f in sorted(os.listdir(zipformer_30m_dir)):
    print(f"  {f}")

In [ ]:
import glob

def find_file(directory, patterns):
    """Find first matching file given list of glob patterns."""
    for pat in patterns:
        matches = glob.glob(os.path.join(directory, pat))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"None of {patterns} found in {directory}")

encoder_30m  = find_file(zipformer_30m_dir, ["encoder*.onnx", "encoder*.int8.onnx"])
decoder_30m  = find_file(zipformer_30m_dir, ["decoder*.onnx"])
joiner_30m   = find_file(zipformer_30m_dir, ["joiner*.onnx", "joiner*.int8.onnx"])
tokens_30m   = find_file(zipformer_30m_dir, ["tokens.txt", "bpe.model", "config.json"])

print(f"encoder : {os.path.basename(encoder_30m)}")
print(f"decoder : {os.path.basename(decoder_30m)}")
print(f"joiner  : {os.path.basename(joiner_30m)}")
print(f"tokens  : {os.path.basename(tokens_30m)}")

recognizer_30m = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=encoder_30m,
    decoder=decoder_30m,
    joiner=joiner_30m,
    tokens=tokens_30m,
    num_threads=4,
    sample_rate=16000,
    feature_dim=80,
    decoding_method="greedy_search",
    provider="cuda",
)
print("ZipFormer-30M loaded.")

def infer_zipformer_30m(audio, sr):
    stream = recognizer_30m.create_stream()
    stream.accept_waveform(sr, audio)
    recognizer_30m.decode_stream(stream)

all_results["ZipFormer-30M"] = measure_rtf(infer_zipformer_30m, samples, "ZipFormer-30M")

del recognizer_30m
import torch; torch.cuda.empty_cache()
import gc; gc.collect()

## 4. Model 2 — ChunkFormer-110M (CTC, CUDA)

In [ ]:
import torch, gc
from chunkformer import ChunkFormerModel

print('Loading ChunkFormer-110M...')
chunkformer_model = ChunkFormerModel.from_pretrained('khanhld/chunkformer-ctc-large-vie')
chunkformer_model.to('cuda')
print('ChunkFormer-110M loaded.')

def infer_chunkformer(audio, sr):
    sf.write(_tmp_wav, audio, sr, subtype='PCM_16')
    chunkformer_model.endless_decode(
        audio_path=_tmp_wav,
        chunk_size=512,
        left_context_size=64,
        right_context_size=32,
        total_batch_duration=30.0,
    )

all_results['ChunkFormer-110M'] = measure_rtf(infer_chunkformer, samples, 'ChunkFormer-110M')

del chunkformer_model
torch.cuda.empty_cache(); gc.collect()

## 5. Model 3 — PhoWhisper-Large (transformers, CUDA)

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

print("Loading PhoWhisper-Large...")
pw_processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-large")
pw_model = WhisperForConditionalGeneration.from_pretrained(
    "vinai/PhoWhisper-large",
    torch_dtype=torch.float16,
).to("cuda")
pw_model.eval()
print("PhoWhisper-Large loaded.")

def infer_phowhisper(audio, sr):
    inputs = pw_processor(audio, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to("cuda", dtype=torch.float16)
    with torch.no_grad():
        pw_model.generate(input_features, language="vi", task="transcribe")

all_results["PhoWhisper-Large"] = measure_rtf(infer_phowhisper, samples, "PhoWhisper-Large")

del pw_model, pw_processor
torch.cuda.empty_cache(); gc.collect()

## 6. Model 4 — VietASR-ZipFormer-68M (sherpa-onnx, CUDA)

In [ ]:
print("Downloading VietASR-ZipFormer-68M ONNX files...")
vietasr_dir = snapshot_download("csukuangfj/sherpa-onnx-zipformer-vi-2025-04-20")
print(f"Downloaded to: {vietasr_dir}")
for f in sorted(os.listdir(vietasr_dir)):
    print(f"  {f}")

In [ ]:
encoder_68m = find_file(vietasr_dir, ["encoder-epoch-*.onnx", "encoder*.onnx"])
decoder_68m = find_file(vietasr_dir, ["decoder-epoch-*.onnx", "decoder*.onnx"])
joiner_68m  = find_file(vietasr_dir, ["joiner-epoch-*.onnx", "joiner*.onnx"])
tokens_68m  = find_file(vietasr_dir, ["tokens.txt", "bpe.model"])

print(f"encoder : {os.path.basename(encoder_68m)}")
print(f"decoder : {os.path.basename(decoder_68m)}")
print(f"joiner  : {os.path.basename(joiner_68m)}")
print(f"tokens  : {os.path.basename(tokens_68m)}")

recognizer_68m = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=encoder_68m,
    decoder=decoder_68m,
    joiner=joiner_68m,
    tokens=tokens_68m,
    num_threads=4,
    sample_rate=16000,
    feature_dim=80,
    decoding_method="greedy_search",
    provider="cuda",
)
print("VietASR-68M loaded.")

def infer_vietasr_68m(audio, sr):
    stream = recognizer_68m.create_stream()
    stream.accept_waveform(sr, audio)
    recognizer_68m.decode_stream(stream)

all_results["VietASR-68M"] = measure_rtf(infer_vietasr_68m, samples, "VietASR-68M")

del recognizer_68m
torch.cuda.empty_cache(); gc.collect()

## 7. Kết quả

In [ ]:
# WER từ paper (bảng VLSP2020-Test-T1)
PAPER_WER = {
    "ZipFormer-30M":   12.29,
    "ChunkFormer-110M": 14.09,
    "PhoWhisper-Large": 13.75,
    "VietASR-68M":      14.45,
}

print(f"{'Model':<22} {'WER (paper)':>12} {'RTF P50':>9} {'RTF P95':>9}")
print("-" * 55)
for name, res in all_results.items():
    wer = PAPER_WER.get(name, float('nan'))
    print(f"{name:<22} {wer:>11.2f}% {res['p50']:>9.4f} {res['p95']:>9.4f}")

In [ ]:
import matplotlib.pyplot as plt

models = list(all_results.keys())
colors = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("ASR RTF — 4 Vietnamese Models (A100, VLSP2020 300 samples)",
             fontsize=12, fontweight="bold")

# Left: P50 + P95 bar chart
ax = axes[0]
x = np.arange(len(models))
w = 0.35
p50s = [all_results[m]["p50"] for m in models]
p95s = [all_results[m]["p95"] for m in models]
bars1 = ax.bar(x - w/2, p50s, w, label="P50", color=colors, alpha=0.85, edgecolor="white")
bars2 = ax.bar(x + w/2, p95s, w, label="P95", color=colors, alpha=0.45, edgecolor="white", hatch="//")
ax.bar_label(bars1, fmt="%.3f", padding=2, fontsize=8)
ax.bar_label(bars2, fmt="%.3f", padding=2, fontsize=8)
ax.axhline(1.0, color="red", linestyle="--", linewidth=1, alpha=0.7, label="RTF=1")
ax.set_title("RTF P50 / P95 — thấp hơn = nhanh hơn")
ax.set_ylabel("RTF")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=12, ha="right")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

# Right: RTF CDF
ax = axes[1]
for i, name in enumerate(models):
    sorted_rtf = sorted(all_results[name]["rtf_values"])
    cdf = np.arange(1, len(sorted_rtf) + 1) / len(sorted_rtf)
    ax.plot(sorted_rtf, cdf, label=name, color=colors[i], linewidth=2)
ax.axvline(1.0, color="red", linestyle="--", linewidth=1, alpha=0.7, label="RTF=1")
ax.set_title("CDF của RTF — đường càng trái càng nhanh")
ax.set_xlabel("RTF")
ax.set_ylabel("Tỷ lệ tích lũy")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/asr_rtf_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/asr_rtf_benchmark.png")

In [ ]:
import csv, json
from datetime import datetime

# Save CSV
csv_path = '/content/asr_rtf_results.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['model', 'wer_paper', 'rtf_p50', 'rtf_p95'])
    for name, res in all_results.items():
        w.writerow([name, PAPER_WER.get(name, ''), f"{res['p50']:.4f}", f"{res['p95']:.4f}"])

# Save JSON (bao gồm full rtf_values để vẽ CDF sau)
json_path = '/content/asr_rtf_results.json'
with open(json_path, 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'rtf_values'}
               | {'rtf_values': v['rtf_values']}
               for k, v in all_results.items()}, f, indent=2)

# Copy lên Drive
!cp {csv_path} {DRIVE_DIR}/asr_rtf_results.csv
!cp {json_path} {DRIVE_DIR}/asr_rtf_results.json
!cp /content/asr_rtf_benchmark.png {DRIVE_DIR}/asr_rtf_benchmark.png

print(f'Saved to Drive: {DRIVE_DIR}/')
print(f'  asr_rtf_results.csv')
print(f'  asr_rtf_results.json')
print(f'  asr_rtf_benchmark.png')